In [2]:
%pip install kagglehub --quiet
import kagglehub
import tqdm as notebook_tqdm
from pathlib import Path

ds1_path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
ds2_path = kagglehub.dataset_download("niyarrbarman/symptom2disease")

print("Dataset 1 root:", ds1_path)
print("Dataset 2 root:", ds2_path)


def _find_csv(root: str, hint: str) -> str:
    """Pick the CSV under `root` whose name contains `hint` (case-insensitive)."""
    candidates = [p for p in Path(root).rglob("*.csv")]
    if not candidates:
        raise FileNotFoundError(f"No CSV under {root}")
    matches = [p for p in candidates if hint.lower() in p.name.lower()]
    chosen = matches[0] if matches else candidates[0]
    return str(chosen)


DS1_CSV = _find_csv(ds1_path, "Final_Augmented")
DS2_CSV = _find_csv(ds2_path, "Symptom2Disease")
print("DS1 CSV:", DS1_CSV)
print("DS2 CSV:", DS2_CSV)


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\LuizAlbertodeAndrade\source\repos\screening_robot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset 1 root: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\dhivyeshrk\diseases-and-symptoms-dataset\versions\1
Dataset 2 root: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\niyarrbarman\symptom2disease\versions\1
DS1 CSV: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\dhivyeshrk\diseases-and-symptoms-dataset\versions\1\Final_Augmented_dataset_Diseases_and_Symptoms.csv
DS2 CSV: C:\Users\LuizAlbertodeAndrade\.cache\kagglehub\datasets\niyarrbarman\symptom2disease\versions\1\Symptom2Disease.csv


In [3]:
%pip install pandas --quiet
import pandas as pd

df_ds1 = pd.read_csv(DS1_CSV)
display(df_ds1.head())
display(df_ds1["diseases"].value_counts())


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


diseases
cystitis                          1219
vulvodynia                        1218
nose disorder                     1218
complex regional pain syndrome    1217
spondylosis                       1216
                                  ... 
foreign body in the nose             1
thalassemia                          1
open wound of the head               1
rocky mountain spotted fever         1
kaposi sarcoma                       1
Name: count, Length: 773, dtype: int64

In [4]:
%pip install numpy --quiet
import re
import numpy as np

def _prettify_symptom(col: str) -> str:
    """Convert e.g. 'shortness_of_breath' -> 'shortness of breath'."""
    return re.sub(r"[_\-]+", " ", col).strip().lower()


def build_ds1(df: pd.DataFrame) -> pd.DataFrame:
    # First column is the disease label; the rest are 0/1 symptom flags.
    disease_col = df.columns[0]
    symptom_cols = [c for c in df.columns[1:]]
    pretty = {c: _prettify_symptom(c) for c in symptom_cols}

    # Vectorised: for each row collect the symptom column names where value == 1.
    sym_matrix = df[symptom_cols].to_numpy(dtype=np.int8)
    rows = []
    for i, disease in enumerate(df[disease_col].astype(str).values):
        active = np.flatnonzero(sym_matrix[i] == 1)
        if active.size == 0:
            continue
        symptoms = ", ".join(pretty[symptom_cols[j]] for j in active)
        rows.append({
            "input":  symptoms,
            "output": disease.strip(),
            "source": "ds1_diseases_symptoms",
        })

    out = pd.DataFrame(rows)
    
    # Drop diseases with <2 rows so stratified split is feasible later.
    counts = out["output"].value_counts()
    keep = counts[counts >= 2].index
    out = out[out["output"].isin(keep)].reset_index(drop=True)
    
    return out


ds1 = build_ds1(df_ds1)
print(f"DS1 rows: {len(ds1):,}  |  unique diseases: {ds1['output'].nunique()}")
ds1.head(3)
ds1["output"].value_counts()



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
DS1 rows: 246,926  |  unique diseases: 754


output
cystitis                                        1219
vulvodynia                                      1218
nose disorder                                   1218
complex regional pain syndrome                  1217
spondylosis                                     1216
                                                ... 
diabetic kidney disease                            2
human immunodeficiency virus infection (hiv)       2
carcinoid syndrome                                 2
rheumatic fever                                    2
open wound of the jaw                              2
Name: count, Length: 754, dtype: int64

In [5]:
df_ds2 = pd.read_csv(DS2_CSV)
#drop unnamed cols
df_ds2 = df_ds2.loc[:, ~df_ds2.columns.str.contains('^Unnamed')]
display(df_ds2.head())
display(df_ds2["label"].value_counts())

,label,text
0,Psoriasis,I have been experiencing a skin rash on my arm...
1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,Psoriasis,I have been experiencing joint pain in my fing...
3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,Psoriasis,"My nails have small dents or pits in them, and..."


label
Psoriasis                          50
Varicose Veins                     50
Typhoid                            50
Chicken pox                        50
Impetigo                           50
Dengue                             50
Fungal infection                   50
Common Cold                        50
Pneumonia                          50
Dimorphic Hemorrhoids              50
Arthritis                          50
Acne                               50
Bronchial Asthma                   50
Hypertension                       50
Migraine                           50
Cervical spondylosis               50
Jaundice                           50
Malaria                            50
urinary tract infection            50
allergy                            50
gastroesophageal reflux disease    50
drug reaction                      50
peptic ulcer disease               50
diabetes                           50
Name: count, dtype: int64

In [6]:
def build_ds2(df: pd.DataFrame) -> pd.DataFrame:    
    df = df.rename(columns={c: c.lower() for c in df.columns})
    assert {"label", "text"}.issubset(df.columns), f"Unexpected columns: {df.columns.tolist()}"
    df = df.dropna(subset=["label", "text"])
    out = pd.DataFrame({
        "input":  df["text"].astype(str).str.strip().values,
        "output": df["label"].astype(str).str.strip().values,
        "source": "ds2_symptom2disease",
    })
    return out


ds2 = build_ds2(df_ds2)
print(f"DS2 rows: {len(ds2):,}  |  unique diseases: {ds2['output'].nunique()}")
ds2.head(3)

DS2 rows: 1,200  |  unique diseases: 24


,input,output,source
0,I have been experiencing a skin rash on my arm...,Psoriasis,ds2_symptom2disease
1,"My skin has been peeling, especially on my kne...",Psoriasis,ds2_symptom2disease
2,I have been experiencing joint pain in my fing...,Psoriasis,ds2_symptom2disease


In [7]:
concat = pd.concat([ds1, ds2], ignore_index=True).reset_index(drop=True)
# normalize output by lowercasing
concat["output"] = concat["output"].str.lower()
print(f"Combined rows: {len(concat):,}  |  unique diseases: {concat['output'].nunique()}")
print("Stable processing index prepared for resumable batching.")

Combined rows: 248,126  |  unique diseases: 767
Stable processing index prepared for resumable batching.


In [8]:
concat["output"].value_counts()

output
pneumonia                                       1262
cystitis                                        1219
vulvodynia                                      1218
nose disorder                                   1218
complex regional pain syndrome                  1217
                                                ... 
diabetic kidney disease                            2
human immunodeficiency virus infection (hiv)       2
carcinoid syndrome                                 2
rheumatic fever                                    2
open wound of the jaw                              2
Name: count, Length: 767, dtype: int64

In [9]:
# Save the combined dataset as a new CSV.
concat.to_csv("combined_diseases_symptoms_2.csv", index=False)